In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!nvidia-smi

Sun Oct 18 14:37:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.11.0+cu128
CUDA available: True
Device: Tesla T4


In [11]:
!git clone https://github.com/Sokach112/DQN.git
%cd DQN

Cloning into 'DQN'...
remote: Enumerating objects: 501, done.
remote: Counting objects: 100% (501/501), done.
remote: Compressing objects: 100% (229/229), done.
remote: Total 501 (delta 252), reused 501 (delta 252), pack-reused 0 (from 0)
Receiving objects: 100% (501/501), 117.00 KiB | 3.66 MiB/s, done.
Resolving deltas: 100% (252/252), done.
/kaggle/working/DQN


In [12]:
import gymnasium as gym
import ale_py
env = gym.make("PongNoFrameskip-v4")
print("Env action space:", env.action_space)
print("Env obs space:", env.observation_space)
env.close()

Env action space: Discrete(6)
Env obs space: Box(0, 255, (210, 160, 3), uint8)


In [13]:
import os

checkpoint_dir = '/kaggle/working/dqn_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

In [14]:
%cd /kaggle/working/DQN
!pip install -r requirements.txt

/kaggle/working/DQN
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 34.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 59.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 67.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 28.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 94.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 5

In [15]:
import hydra
print("Hydra OK:", hydra.__version__)

Hydra OK: 1.3.2


In [16]:
%cd /kaggle/working/DQN
!python main.py \
  total_epoch=2500000 interact_per_epoch=1 \
  env.num_testing_envs=1 \
  algorithm.buffer_size=200000 algorithm.learning_starts=20000 \
  algorithm.learning_rate=1e-4 algorithm.batch_size=32 \
  algorithm.epsilon_timestep=250000 algorithm.end_epsilon=0.1 \
  algorithm.target_update_interval=1000 \
  test_interval=50000 test_episodes=10 save_interval=100000

/kaggle/working/DQN
2026-10-18 16:05:55 [INFO] ---------------NumExpr defaulting to 4 threads.---------------
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Parameters: env:
  name: PongNoFrameskip-v4
  num_training_envs: 1
  num_testing_envs: 1
  frame_stack: 4
algorithm:
  name: DQN
  activation: leaky_relu
  buffer_name: ReplayBuffer
  buffer_size: 200000
  learning_rate: 0.0001
  gamma: 0.99
  optimizer: Adam
  start_epsilon: 1
  end_epsilon: 0.1
  epsilon_schedule: linear
  learning_starts: 20000
  epsilon_timestep: 250000
  use_target: